# Two ways to delegate

**Scenario:** a sequencing lab finishes three variant calling runs a night. Each writes a long shard
by shard log. A release owner needs one word out of it. Ship or hold.

Think of it as a work order and a conversation. LangGraph hands a worker a form with named boxes and
takes the form back. CrewAI hands a worker a brief and trusts the reply. Both run here, same job,
same model.

## Mechanics

| Piece | LangGraph | CrewAI |
|---|---|---|
| the unit of work | a node function | an `Agent` paired with a `Task` |
| what one worker gets | the `Send` payload, shaped by its own state class | the task text, plus earlier task output |
| what crosses back | only the channels the parent state declares | the task's output text |
| how a shape is declared | the parent state class | `output_pydantic` or `guardrail` on the task, if you add one |
| where the model is set | your own call, inside the node | an `LLM` object on the agent |
| fan out over many items | one `Send` per item from a single edge | more tasks, or one task that loops |

`START` is an edge source like any other, so `add_edge(START, node)` is how a graph begins. A channel
marked with a reducer such as `operator.add` is what lets several workers write to it at once.

## The picture

![One job, two boundaries: a declared parent state and a task output](images/two-ways-to-delegate.svg)

The work is identical. What differs is what you get to declare about the crossing.

## The cost

```
tokens = sum(prompt + completion) over every call the framework made
```

The framework does not do the reading, the model does, so the bill should land close either way.

## The failure

Three runs, and a log long enough to be a problem.

In [1]:
def run_log(run_id, variants):
    """One variant calling log, the way the pipeline writes it."""
    lines = [f"RUN {run_id} caller=deepvariant ref=GRCh38"]
    for shard in range(1, 25):
        lines.append(f"  shard {shard:02d} reads=1841203 mapq_ok=0.981 dup=0.037 elapsed=91s")
    lines.append(f"  called: {variants} variants pass filter, 4 flagged low depth")
    lines.append("  warn: contig chrM coverage below threshold on 2 shards")
    return "\n".join(lines)


LOGS = {"RUN-9101": run_log("RUN-9101", 41822),
        "RUN-9102": run_log("RUN-9102", 39004),
        "RUN-9103": run_log("RUN-9103", 40517)}
print(f"{len(LOGS)} runs, {len(LOGS['RUN-9101'])} characters of log each")

3 runs, 1621 characters of log each


One helper makes every call, so both frameworks are counted the same way.

In [2]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("05-subagent-delegation/02-two-ways-to-delegate")
BILLED = []

READER = ("You read variant calling logs. Reply with one line: "
          "run=<id>; variants=<n>; verdict=<pass|review>.")
RELEASE = "You own the release. Given short run reports, reply with one word: ship or hold."


def ask(system, user, cap=80):
    """One call, counted, so both frameworks are compared on the same basis."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=cap,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}])
    BILLED.append(reply.usage.prompt_tokens + reply.usage.completion_tokens)
    return (reply.choices[0].message.content or "").strip()

Here is the first LangGraph most people write. One state class for the whole graph, because that is
the shortest thing that works.

In [3]:
import json
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class FlatState(TypedDict):
    """One state for everybody, so a worker's scratch is the parent's memory."""

    run_ids: list
    logs: Annotated[list, operator.add]
    reports: Annotated[list, operator.add]

The worker reads its log and writes both the log and its report back. That looks fine while you are
typing it.

In [4]:
PARENT_STATE_BUDGET = 1000

flat = StateGraph(FlatState)
flat.add_node("read_run", lambda s: {"logs": [LOGS[s["run_id"]]],
                                     "reports": [ask(READER, LOGS[s["run_id"]])]})
flat.add_conditional_edges(
    START, lambda s: [Send("read_run", {"run_id": r}) for r in s["run_ids"]], ["read_run"])
flat.add_edge("read_run", END)

flat_out = flat.compile().invoke({"run_ids": list(LOGS), "logs": [], "reports": []})
flat_size = len(json.dumps(flat_out))
for line in flat_out["reports"]:
    print(f"  {line}")
print(f"\nparent state after the run: {flat_size} characters")
assert flat_size <= PARENT_STATE_BUDGET, (
    f"the parent state is carrying {flat_size} characters, "
    f"including {len(flat_out['logs'])} raw logs")

  run=RUN-9101; variants=41822; verdict=review
  run=RUN-9102; variants=39004; verdict=review
  run=RUN-9103; variants=40517; verdict=review

parent state after the run: 5169 characters


AssertionError: the parent state is carrying 5169 characters, including 3 raw logs

## The diagnosis

The reports are correct and the state is enormous.

`logs` is a channel on the parent state, so a worker writing to it writes into the thing the parent
keeps. A reducer does not care where a value came from. It appends.

One state class means the worker and the parent share a memory, which is the opposite of delegating.
LangGraph draws the line where you declare one, and nowhere else.

## The fix

Give the worker its own state. The `Send` payload is the whole of what a worker gets.

In [5]:
class WorkerState(TypedDict):
    """Everything one worker is given, and the only thing it can see."""

    run_id: str
    log: str

Then the parent state, which is the real fix. No channel for a log, so no worker can leave one.

In [6]:
class ParentState(TypedDict):
    """What the parent is allowed to hold. Notice what is missing."""

    run_ids: list
    reports: Annotated[list, operator.add]
    release: str

The worker is an ordinary function over `WorkerState`. It returns one key, and that key is the only
thing that crosses back.

In [7]:
def read_run(state: WorkerState) -> dict:
    """Read one log. The log lives in the worker state and dies with it."""
    return {"reports": [ask(READER, state["log"])]}

Wire it. `add_edge(START, "fan_out")` gives the graph one entry, and the fan out node builds the
`Send` calls.

In [8]:
BILLED.clear()
graph = StateGraph(ParentState)
graph.add_node("fan_out", lambda s: {})
graph.add_node("read_run", read_run)
graph.add_node("decide", lambda s: {"release": ask(RELEASE, "\n".join(s["reports"]), 20)})
graph.add_edge(START, "fan_out")
graph.add_conditional_edges(
    "fan_out",
    lambda s: [Send("read_run", {"run_id": r, "log": LOGS[r]}) for r in s["run_ids"]],
    ["read_run"])
graph.add_edge("read_run", "decide")
graph.add_edge("decide", END)

final_state = graph.compile().invoke({"run_ids": list(LOGS), "reports": [], "release": ""})
state_chars = len(json.dumps(final_state))
graph_tokens = sum(BILLED)
print(f"release : {final_state['release']}")
print(f"before  : {flat_size} characters of parent state")
print(f"after   : {state_chars} characters of parent state")

release : hold
before  : 5169 characters of parent state
after   : 225 characters of parent state


Same logs, same reports, and the parent forgot them because it could never hold them.

Now the same job as a crew. CrewAI reaches for its own model client, which our recordings do not
capture, so its model port goes through the vault client instead. That is how this runs with no key.
The environment variables switch off consent prompts, telemetry and tracing, which otherwise phone
home and can stop to ask a question.

In [9]:
import os

os.environ["CREWAI_TESTING"] = "true"           # no first run consent prompt, no browser
os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

from crewai import Agent, BaseLLM, Crew, Process, Task

CREW_BILL, CROSSED, REASONS = [], [], []


class VaultLLM(BaseLLM):
    """CrewAI's model port, wired to the vault client so a crew records and replays."""

    def call(self, messages, tools=None, callbacks=None, available_functions=None,
             from_task=None, from_agent=None, response_model=None):
        clean = [{"role": m["role"], "content": m["content"]} for m in messages]
        reply = client.chat.completions.create(
            model=self.model, max_tokens=400, messages=clean)
        CREW_BILL.append(reply.usage.prompt_tokens + reply.usage.completion_tokens)
        CROSSED.append(json.dumps(clean))
        REASONS.append(reply.choices[0].finish_reason)
        return (reply.choices[0].message.content or "").strip()

Two agents, one task each, and the second takes the first as context. The release agent gets the rule
the graph gave its decide node. The kickoff runs on a worker thread, because a notebook already has
an event loop and CrewAI's synchronous path refuses to start inside one.

In [10]:
from concurrent.futures import ThreadPoolExecutor

llm = VaultLLM(model=model_for("default"))
reader = Agent(role="Log reader", goal="Report variants and a verdict for each run",
               backstory="You read variant calling logs.", llm=llm,
               verbose=False, max_iter=1, allow_delegation=False)
gate = Agent(role="Release gate", goal="Ship or hold the batch", backstory=RELEASE,
             llm=llm, verbose=False, max_iter=1, allow_delegation=False)
read = Task(description="For each log below reply with one line "
                        "run=<id>; variants=<n>; verdict=<pass|review>.\n\n"
                        + "\n\n".join(LOGS.values()),
            expected_output="three lines", agent=reader)
decide = Task(description="Reply with one word, ship or hold.",
              expected_output="one word", agent=gate, context=[read])
crew = Crew(agents=[reader, gate], tasks=[read, decide],
            process=Process.sequential, verbose=False)

with ThreadPoolExecutor(max_workers=1) as pool:
    crew_out = pool.submit(crew.kickoff).result()
print(f"crew release: {crew_out.raw}")

crew release: ship


Both produced a release word. Read every line of the comparison.

In [11]:
print(f"{'framework':10} {'calls':>6} {'tokens':>8}   what the parent is left holding")
print(f"{'langgraph':10} {len(BILLED):>6} {graph_tokens:>8}   "
      f"{state_chars} characters of state it declared")
print(f"{'crewai':10} {len(CREW_BILL):>6} {sum(CREW_BILL):>8}   "
      f"{len(CROSSED[1])} characters in the second prompt")
print(f"\ncrew finish reasons                  : {REASONS}")
print(f"raw log inside the first crew prompt : {'shard 01 reads' in CROSSED[0]}")
print(f"raw log inside the second crew prompt: {'shard 01 reads' in CROSSED[1]}")

framework   calls   tokens   what the parent is left holding
langgraph       4     3034   225 characters of state it declared
crewai          2     3804   1180 characters in the second prompt

crew finish reasons                  : ['length', 'stop']
raw log inside the first crew prompt : True
raw log inside the second crew prompt: True


The bill is the least interesting line. In live runs where the reader behaved, the crew came in near
3100 tokens against the graph's 3034. They cost the same. The gap above is a reply thrown away.

Now the other three lines. The crew's first reply finished on `length`, so it was cut off at the token
cap. It was cut off because the reader returned its own input instead of a summary, and CrewAI passed
the truncated log on as the task output. A raw log crossed, and the release agent, holding half a log
instead of three reports, answered ship where the graph answered hold.

It does not happen every time. Across ten live runs the reader echoed its input four times. What
matters is not four. It is that nothing in the crew disagreed on any of the ten.

`ParentState` has no channel for a log, so the graph could not do this. CrewAI offers the same answer
in `output_pydantic` or a `guardrail`. Neither framework is the safer one. The safety comes from the
declaration, and only one of them made you write it first.

## The gate

Because the parent state is declared, it can be tested.

In [12]:
def test_the_parent_holds_only_what_it_declared():
    body = json.dumps(final_state)
    assert "shard 01 reads" not in body, "a raw log line is sitting in the parent state"
    assert set(final_state) == set(ParentState.__annotations__), (
        f"the parent state grew a channel: {sorted(final_state)}")


test_the_parent_holds_only_what_it_declared()
print("gate holds: the parent carries the channels it declared, and nothing else")

gate holds: the parent carries the channels it declared, and nothing else


Add `logs` back to `ParentState` and have `read_run` return it, and this test fails on the first line.

### Enterprise exploration

- Three runs fan out fine. What happens at three hundred, and where does the concurrency limit go?
- A worker fails halfway. What does the parent see in each, and what does resuming cost?
- The logs are gone from both. An auditor asks what the release rested on. Where were they kept, for
  how long, and who may read them?
- Four of ten runs pushed a truncated log across. What would you measure in production to see that?

### Key takeaways

- A shared state is not a boundary. The worker writes straight into the parent's memory.
- `Send` plus a worker state class draws the line, and the parent state says what may cross back.
- CrewAI passes the task output, not the prompt. Nothing declares what an output may hold, so a
  truncated log crossed and flipped the answer.
- The token cost matched. The difference is what you can assert afterwards.